# simple-steps-core Interactive Walkthrough

This notebook reproduces the same core flow as `runme.py`, but explains each concept as you execute cells.

What you will learn:
- How `OperationRegistry` turns Python functions into operations.
- The difference between deferred calls (`ToolCall`) and immediate execution (`.run(...)`).
- How `Workflow` and `CoreEngine` cooperate during execution.
- Why assertions are useful as lightweight correctness checks.
- How to run parameterized experiments interactively with widgets.

In [1]:
from simple_steps_core import CoreEngine, OperationRegistry, StepStatus, ToolCall, Workflow

# Optional interactivity. The notebook still works without ipywidgets.
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    widgets = None
    from IPython.display import Markdown

print("Imports ready.")
print(f"ipywidgets available: {HAS_WIDGETS}")

Imports ready.
ipywidgets available: False


## 1) Register Operations

An **operation** is a regular Python function registered under an ID.

`OperationRegistry` stores:
- The callable function.
- A serializable operation definition (name, params, defaults).
- A dual-mode wrapper (`Operation`) that supports deferred and immediate modes.

Deferred mode is useful when authoring workflows. Immediate mode is useful for quick direct tests.

In [2]:
registry = OperationRegistry()


def make_list(n: int) -> list[int]:
    return list(range(n))


def total(data: list[int]) -> int:
    return sum(data)


make_list_op = registry.register("make_list", make_list, description="Create [0..n-1]")
total_op = registry.register("total", total, description="Sum a list of ints")

print("Registered operations:")
for definition in registry.list_definitions():
    print(f"- {definition.operation_id}: {[p.name for p in definition.params]}")

Registered operations:
- make_list: ['n']
- total: ['data']


## 2) Deferred vs Immediate Execution

Calling a registered operation normally returns a **ToolCall** (deferred).

- Deferred: `make_list_op(n=5)` returns a data structure describing what to run later.
- Immediate: `make_list_op.run(n=5)` executes now and returns a Python value.

This pattern separates **authoring** a workflow from **executing** it.

In [3]:
deferred_call = make_list_op(n=5)
immediate_value = make_list_op.run(n=5)

print("Deferred type:", type(deferred_call).__name__)
print("Deferred payload:", deferred_call)
print("Immediate value:", immediate_value)

assert isinstance(deferred_call, ToolCall)
assert immediate_value == [0, 1, 2, 3, 4]
print("Deferred and immediate checks passed.")

Deferred type: ToolCall
Deferred payload: operation_id='make_list' arguments={'n': 5}
Immediate value: [0, 1, 2, 3, 4]
Deferred and immediate checks passed.


## 3) Workflow + Session Context

`Workflow` stores step formulas and executes them in insertion order.

`CoreEngine` handles one tool call at a time by:
1. Validating arguments against operation definitions.
2. Resolving references (when present).
3. Executing the underlying Python function.
4. Storing result under a session-scoped reference.

In this walkthrough, we run one workflow step (`make_list`) and then run `total` directly through the engine, just like `runme.py`.

In [4]:
engine = CoreEngine(registry)
workflow = Workflow(engine, session_id="notebook-demo")

workflow["step1"] = make_list_op(n=5)
steps = workflow.run()
step1 = steps[0]

assert step1.status is StepStatus.COMPLETED
assert step1.output is not None
assert step1.output.value == [0, 1, 2, 3, 4]

print("Workflow step completed")
print("step1 output:", step1.output.value)
print("step1 output ref:", step1.output.ref)

Workflow step completed
step1 output: [0, 1, 2, 3, 4]
step1 output ref: notebook-demo__7d53435e288a43a2be20d12ca4467adc


In [5]:
ref_id, total_value = engine.execute(
    ToolCall(operation_id="total", arguments={"data": step1.output.value}),
    workflow.context,
)

assert ref_id.startswith("notebook-demo__")
assert total_value == 10

print("Engine direct execute succeeded")
print("total:", total_value)
print("stored ref:", ref_id)

Engine direct execute succeeded
total: 10
stored ref: notebook-demo__a35534aaaba14377840d3b8d5b70bc82


## 4) Interactive Experiment

Use the sliders/text below to re-run the same pipeline with different values.

If `ipywidgets` is unavailable, the fallback helper still lets you call the function directly.

In [6]:
def run_experiment(n: int = 5, session_id: str = "interactive") -> dict:
    local_registry = OperationRegistry()

    def make_list_local(n: int) -> list[int]:
        return list(range(n))

    def total_local(data: list[int]) -> int:
        return sum(data)

    make_list_local_op = local_registry.register("make_list", make_list_local)
    local_registry.register("total", total_local)

    local_engine = CoreEngine(local_registry)
    local_workflow = Workflow(local_engine, session_id=session_id)
    local_workflow["step1"] = make_list_local_op(n=n)
    local_step1 = local_workflow.run()[0]

    total_ref, total_out = local_engine.execute(
        ToolCall(operation_id="total", arguments={"data": local_step1.output.value}),
        local_workflow.context,
    )

    result = {
        "n": n,
        "session_id": session_id,
        "step1": local_step1.output.value,
        "total": total_out,
        "total_ref": total_ref,
    }
    return result


# Non-widget fallback usage:
print(run_experiment(n=6, session_id="fallback-demo"))

{'n': 6, 'session_id': 'fallback-demo', 'step1': [0, 1, 2, 3, 4, 5], 'total': 15, 'total_ref': 'fallback-demo__cc804089c9974e32a48a99be0bef27c9'}


In [7]:
if HAS_WIDGETS:
    n_slider = widgets.IntSlider(value=5, min=1, max=25, step=1, description="n")
    session_text = widgets.Text(value="widget-demo", description="session")
    run_btn = widgets.Button(description="Run pipeline", button_style="success")
    out = widgets.Output()

    def on_click(_):
        with out:
            out.clear_output()
            result = run_experiment(n=n_slider.value, session_id=session_text.value)
            print(result)

    run_btn.on_click(on_click)
    display(widgets.VBox([n_slider, session_text, run_btn, out]))
else:
    print("ipywidgets is not installed. Install it to enable UI controls:")
    print("  .venv/bin/python -m pip install ipywidgets")

ipywidgets is not installed. Install it to enable UI controls:
  .venv/bin/python -m pip install ipywidgets
